# CFPB complaints — data exploration

Inputs: `data/raw_complaints.json` (12,000 sampled complaints with narratives) and the
splits written by `src/preprocessing/build_dataset.py`.

Questions this notebook answers, in the order section 15 asks them:

1. What is the product distribution, before choosing any categories?
2. Are the narratives long enough to classify, and how long in *tokens*?
3. How much duplication is there, and where does it come from?
4. Do the splits come out balanced and leak-free?

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style='whitegrid')
pd.set_option('display.width', 120)

raw = pd.read_json('../data/raw_complaints.json', dtype={'complaint_id': str})
print(raw.shape)
raw.head(3)

## 1. Product distribution

The sample is capped at 2,000 per category by `fetch_cfpb_data.py`. The interesting column is
`product`: CFPB renamed two categories during 2023, so eight product strings map to six labels.

In [ ]:
from sys import path
path.append('../src/preprocessing')
from fetch_cfpb_data import CATEGORIES

product_to_label = {p: l for l, ps in CATEGORIES.items() for p in ps}
raw['label'] = raw['product'].map(product_to_label)

print(raw.groupby('label')['product'].value_counts().to_string())

## 2. Narrative length

`max_length=256` is fixed in `export_onnx.py` and `metadata.json`, so the question is what
fraction of complaints gets truncated there. Measured with the same tokenizer the model uses.

In [ ]:
raw['text'] = raw['complaint_what_happened'].fillna('').str.strip()
raw['n_words'] = raw.text.str.split().str.len()

print(raw.n_words.describe(percentiles=[.5, .75, .9, .95]).round(1).to_string())
print('\nempty or very short (<20 chars):', (raw.text.str.len() < 20).sum())

In [ ]:
from tokenizers import Tokenizer

tok = Tokenizer.from_file('../model/tokenizer.json')
raw['n_tokens'] = [len(e.ids) for e in tok.encode_batch(raw.text.tolist())]

MAX_LEN = 256
print(raw.n_tokens.describe(percentiles=[.5, .75, .9, .95]).round(1).to_string())
print(f'\ntruncated at {MAX_LEN} tokens: {(raw.n_tokens > MAX_LEN).mean():.1%} of complaints')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(raw.n_tokens.clip(upper=1000), bins=60, ax=ax)
ax.axvline(MAX_LEN, color='crimson', ls='--', label=f'max_length = {MAX_LEN}')
ax.set(xlabel='tokens per complaint (clipped at 1000)', ylabel='complaints')
ax.legend()
fig.tight_layout()
fig.savefig('../diagrams/token_length_distribution.png', dpi=150)

In [ ]:
# Median length per class: mortgage complaints are the longest, credit reporting the shortest.
raw.groupby('label')[['n_words', 'n_tokens']].median().round(0).sort_values('n_tokens')

## 3. Duplication

Credit-repair template letters get filed verbatim many times. Left in, the same text would land
in train and test, inflating every score — `build_dataset.py` drops them.

In [ ]:
dupes = raw[raw.duplicated('text', keep=False) & (raw.text.str.len() >= 20)]
print('rows sharing their narrative with another row:', len(dupes))
print(dupes.label.value_counts().to_string())

ambiguous = raw.groupby('text').label.nunique()
print('\ntexts filed under more than one label:', (ambiguous > 1).sum())

In [ ]:
# The most-repeated narrative, and how often it appears.
top = dupes.text.value_counts().head(1)
print(f'{top.iloc[0]} copies:\n')
print(top.index[0][:400])

## 4. Sample narratives

`XXXX` is CFPB's own redaction of personal details — it survives into training as ordinary tokens.

In [ ]:
for label, group in raw[raw.text.str.len() > 200].groupby('label'):
    print(f'--- {label} ---')
    print(group.text.iloc[0][:300].replace('\n', ' '), '\n')

## 5. The splits

Written by `build_dataset.py`: 1,350 per class, stratified 80/10/10, seed 42.

In [ ]:
splits = {s: pd.read_parquet(f'../data/{s}.parquet') for s in ['train', 'val', 'test']}
print({s: len(d) for s, d in splits.items()})

pd.DataFrame({s: d.label.value_counts(normalize=True).round(3) for s, d in splits.items()})

In [ ]:
ids = {s: set(d.complaint_id) for s, d in splits.items()}
texts = {s: set(d.text) for s, d in splits.items()}
pairs = [('train', 'val'), ('train', 'test'), ('val', 'test')]
print('id overlap:  ', {f'{a}/{b}': len(ids[a] & ids[b]) for a, b in pairs})
print('text overlap:', {f'{a}/{b}': len(texts[a] & texts[b]) for a, b in pairs})

### Takeaways

- Six categories, balanced at 1,350 each, and the renamed CFPB products are merged.
- Most complaints fit inside `max_length=256`; the tail that does not is dominated by mortgage.
- Template duplication is real and concentrated in credit reporting, and is removed before splitting.
- Splits are proportional and share no complaint id or narrative text.